### Oncoradqwen: Training data generation

Set up environment (including `assume --env` credentials if required)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from datagen import BedrockBatchGenerator, MESARuntimeBatchGenerator, TrainingDataUploader, DocumentLoader, BatchOutputCorrector
from oncoradschema.prompt_builder import PromptBuilder
from oncoradschema import Schema

In [ ]:
pb = PromptBuilder()

#### Download documents from batch

In [ ]:
DocumentLoader.list_available_document_batches()

In [ ]:
gen = MESARuntimeBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    schema=Schema,
    schema_name="oncoradschema",
    document_batches=["oncorad_sonnet45-2026-09-25-001.tar.gz"],
    base_url=os.environ["BASE_URL"],
    username=os.environ["USERNAME"],
    password=os.environ["PASSWORD"],
    model_name="qwen3.5"
)

In [ ]:
gen.get_document_files_count()

#### Benchmark

##### Start batch generation (MESA Runtime)

In [ ]:
gen.generate_via_batch(100, os.environ["BUCKET"])

##### Download and parse batch outputs (MESA)

In [ ]:
gen.extract_batch_output(os.environ["BUCKET"])

##### Start batch generation

In [ ]:
gen_aws = BedrockBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    schema=Schema,
    schema_name="oncoradschema",
    document_batches=["oncorad_sonnet45-2026-09-25-001.tar.gz"],
    model_name="sonnet4",
)

In [ ]:
gen_aws.generate_via_batch(100, os.environ["BUCKET"], os.environ["BEDROCK_EXECUTION_ROLE"])

##### Download and parse batch outputs

In [ ]:
gen_aws.extract_batch_output(os.environ["BUCKET"])

#### Start datagen

In [ ]:
gen_full = MESARuntimeBatchGenerator(
    system_prompt=pb.build_datagen_prompt(),
    schema=Schema,
    schema_name="oncoradschema",
    document_batches=["oncorad_sonnet45-2026-09-25-001.tar.gz", "oncorad_gemini_2026-09-23.tar.gz", "normal-batch-2025-12-31-001.tar.gz"],
    base_url=os.environ["BASE_URL"],
    username=os.environ["USERNAME"],
    password=os.environ["PASSWORD"],
    model_name="qwen3.5"
)

In [ ]:
gen_full.get_document_files_count()

In [ ]:
gen_full.generate_via_batch(10000, os.environ["BUCKET"])

In [ ]:
gen_full.extract_batch_output(os.environ["BUCKET"])

#### Correct batch outputs

In [ ]:
corrector = BatchOutputCorrector(Schema, Path("anthropic_batch_job.jsonl.out"))

In [ ]:
corrector.correct_with_claude(Path("prompts/correction.txt"), pb.build_datagen_prompt())

In [ ]:
corrector.correct_with_rules(Path("rules/correction.yaml"))

#### Upload formatted document:schema pairs as training data

In [ ]:
s3_uri = TrainingDataUploader.upload(
    schema=Schema,
    schema_name="oncoradschema",
    system_prompt=pb.build_main_prompt(),
    short_description="oncoradqwen-batch",
    long_description="Training data for Oncoradqwen",
    input_folder=Path("./data/trainingdata"),
)